## Langchain v1.0을 이용해서 챗봇 구현하기

지난 시간에 한 LLM 챗봇을 다시 만들어 봅시다!

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver


import os
from dotenv import load_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(dotenv_path="../.env")

# 모델 이름
model_name = "gpt-5.4-mini"

# LLM 정의
model = init_chat_model(
    model=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


그냥 모델은 도구를 사용할 수 없기 때문에 아래와 같은 질문들에 제대로 대답할 수 없습니다.

In [4]:
response = model.invoke("오늘 서울 날씨는 어때?")
print(response.content)

실시간 날씨는 제가 직접 조회할 수 없어서 **오늘 서울의 현재 날씨**를 바로 알려드리진 못해요.  
대신 아래 방법으로 빠르게 확인할 수 있어요:

- **네이버 날씨** 또는 **카카오맵**
- **기상청 날씨누리**
- 휴대폰 기본 날씨 앱에서 **서울** 검색

원하시면 제가 대신  
1) **오늘 서울 날씨를 확인하는 가장 빠른 방법**을 알려드리거나,  
2) **서울의 평균적인 4월/현재 계절 날씨 옷차림**을 추천해드릴게요.


그렇기 때문에 2가지가 필요합니다.

1. 모델이 사용할 수 있는 도구(Tool)
2. 모델이 도구를 사용할 수 있도록 에이전트로 변환.

#### Tool

Tool이란, LLM이 사용할 수 있는 도구를 말합니다. @Tool 데코레이션을 사용해 구현할 수 있습니다.

LLM에게 어떤 경우에 어떻게 사용할 수 있는지 자세히 설명해주는 것이 좋습니다.

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 기능""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

`create_agent()`를 통해 LLM을 agent로 변환해 줄 수 있습니다!

In [7]:
# agent로 변환 : 이제 llm이 도구를 스스로 쓸 수 있게 됩니다.
agent = create_agent(
    model,
    tools=[get_weather],
    system_prompt="너는 기상캐스터로써, 사용자가 물어보는 지역에 대한 날씨 정보를 알려줘야 한다."
)

다시 한 번 날씨를 물어볼까요?

In [21]:
messages = [
    {"role": "user", "content": "오늘 서울의 날씨는 어떤가요?"},
]

response = agent.invoke(
    {"messages": messages}
)

print("모델의 tool calling :", response['messages'][1].tool_calls)
print("도구 사용 결과 : ", response['messages'][2])
print('\n')
print(response['messages'][3].content)

모델의 tool calling : [{'name': 'get_weather', 'args': {'location': '서울'}, 'id': 'call_HqHoNUxUsX98PDvRwdXwP4MC', 'type': 'tool_call'}]
도구 사용 결과 :  content="It's sunny in 서울" name='get_weather' id='06348c44-60ba-4715-b7ac-cd261c577c9c' tool_call_id='call_HqHoNUxUsX98PDvRwdXwP4MC'


오늘 서울은 **맑은 날씨**입니다.  
대체로 **햇볕이 좋은 편**이에요.

원하시면 제가 **기온, 강수확률, 바람**까지 같이 알려드릴게요.


모델이 `get_weather()` 함수를 스스로 호출하고 결과를 확인한 뒤에 '서울의 날씨는 **맑음**' 이라고 대답한 것을 확인할 수 있습니다!

이렇게 도구를 정의하고, 설명만 LLM이 이해할 수 있도록 작성을 해둔다면 LLM이 스스로 도구를 사용합니다!

실습1 : 계산기를 사용하는 LLM

예로부터 LLM이 기초 연산에 약점을 보이는 문제가 많았었는데요,

직접 사칙연산을 수행하는 함수를 구현하고 LLM이 tool로 활용해 연산을 수행할 수 있도록 해봅시다!

`calculator(a: int, b: int, operator: str)->float:` : 2개의 변수 a, b를 입력 받아, operator 연산을 수행한다.

In [22]:
@tool
def calculator(a: int, b: int, operator: str) -> float:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에 
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "plus":
        return a + b
    if operator == "minus":
        return a - b
    if operator == "multiply":
        return a * b
    if operator == "divide":
        return a / b


agent = create_agent(
    model,
    tools=[calculator],
    system_prompt="너는 친절한 계산기야. 사용자가 질문한 사칙연산을 빠르고 정확하게 답변해야 해."
)

questions = [
    [{"role": "user", "content": "3178 + 254의 계산결과는?"},],
    [{"role": "user", "content": "5137 - 33은?"},],
    [{"role": "user", "content": "223 x 317이 뭐야?"},],
    [{"role": "user", "content": "57을 8로 나누면 뭐야?"},],
]

for question in questions:
    response = agent.invoke(
        {"messages": question}
    )
    print(response['messages'][-1].content)

3178 + 254 = 3432
5104
223 × 317 = 70,691입니다.
57 ÷ 8 = 7.125 입니다.


도구를 여러개 쥐어주고, 각 상황에 적절한 도구를 호출하는 지도 확인해 봅시다!

In [30]:
@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 함수""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

@tool
def calculator(a: int, b: int, operator: str) -> float:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에 
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "plus":
        return a + b
    if operator == "minus":
        return a - b
    if operator == "multiply":
        return a * b
    if operator == "divide":
        return a / b

@tool
def get_temperature(location: str) -> str:
    """특정 장소의 기온 정보를 알려주는 함수"""
    return f"{location}의 기온은 17도 입니다."

agent = create_agent(
    model,
    tools=[get_temperature, get_weather, calculator],
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "오늘 부산의 날씨와 기온을 알려줘. 기온 알려줄 때는 10도를 더해서 알려줘."}
    ]}
)

In [40]:
from langchain_core.messages import AIMessage

for message in response['messages']:
    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print("tool calling :", tool_call['name'])
    if message.content:
        print("content :", message.content)

content : 오늘 부산의 날씨와 기온을 알려줘. 기온 알려줄 때는 10도를 더해서 알려줘.
tool calling : get_weather
tool calling : get_temperature
content : It's sunny in 부산
content : 부산의 기온은 17도 입니다.
tool calling : calculator
content : 27
content : 오늘 부산의 날씨는 **맑음**입니다.  
기온은 **17도**이고, 요청하신 대로 **10도를 더하면 27도**입니다.


#### Agents.md

시스템 프롬프트란? : AI의 행동지침을 정의하는 프롬프트.

일반적으로 에이전트의 행동 방침은 코드와 분리하여 `agents.md`라는 파일에 작성을 하고 로드하는 식으로 구현합니다.

이는 아래와 같은 이점들 때문입니다.

1. 코드가 깔끔해진다 : 코드 안에 프롬프트가 길게 들어가면 가독성이 떨어짐.
2. 수정이 편하다 : 프롬프트 바꿀 때마다 코드를 수정할 필요 없이 `agents.md`만 수정하면 됨.
3. 협업/버전 관리가 쉬움 : 프롬프트 변경 이력을 확인하기 용이하고, 개발자가 아닌 사람도 쉽게 편집할 수 있음.

In [ ]:
# agent = create_agent(
#     model,
#     tools=[get_weather, get_temperature, calculator],
#     system_prompt=
#     """
#     ## 역할
#     - 사용자의 질문에 정확하고 간결하게 답하는 AI 어시스턴트

#     ## 규칙
#     - 핵심 내용만 짧게 전달하고, 불필요한 부연 설명은 생략
#     - 도구 실행 결과는 그대로 인용하며, 임의로 수정하지 않기
#     - 계산이나 조회가 필요한 경우 반드시 도구를 사용하고 추측으로 답하지 않기
#     - 불확실한 내용은 단정하지 않기
#     """
# )

# 이랬던 코드가...

In [45]:
# 이렇게 짧게 바뀜

with open("agents.md", encoding="utf-8") as f:
    system_prompt = f.read()

tools = [get_weather, get_temperature, calculator]

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_prompt
)

#### 메모리 관리

앞서도 살펴봤듯이, 기본적으로는 이전의 대화 내역을 모두 모델에게 입력하는 식으로 모델의 메모리를 유지합니다.

In [44]:
messages = [
    {"role": "user", "content": "지금부터 네 이름은 트럼프야."},
    {"role": "assistant", "content": "알겠습니다! 지금부터는 저를 트럼프라고 불러주세요! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "트럼프, 나 오늘 주식을 다 잃었어. 어떡하지?"}
]

response = agent.invoke(
    {"messages": messages}
)

print(response['messages'][-1].content)

많이 힘드셨겠어요. 지금은 **추가 매매를 멈추고** 먼저 상황을 정리하는 게 중요합니다.

- **오늘은 거래 앱/차트를 닫기**
- **손실 금액을 정확히 확인**
- **현금 유동성부터 확보**
- **빚으로 투자한 게 있으면 상환 계획 확인**
- **다음엔 감당 가능한 금액만 투자**하기

감정이 너무 크게 흔들리면 혼자 버티지 말고 **가까운 사람에게 바로 말하세요**.  
만약 지금 **스스로를 해칠 생각**이 조금이라도 든다면, **즉시 112/119 또는 자살예방상담전화 109**에 연락하세요.

원하시면 제가 바로  
1) **손실 정리 체크리스트**  
2) **내일 할 일 3가지**  
로 짧게 정리해드릴게요.


하지만 귀찮죠?

직접 대화를 전부 `messages` 리스트에 기록하고, 다시 꺼내오고...

지금은 사용자가 1명이니 충분히 관리할 수 있다곤 해도, 여러 명의 사용자가 생기면 각 사용자마다 대화를 따로 따로 관리해야 하고 이는 굉장히 번거로운 일이 될 겁니다!

당연히 LangChain에서는 이를 편리하게 할 수 있는 메모리 기능을 따로 제공하고 있습니다. 바로 `InMemorySaver`죠!

In [ ]:
memory = InMemorySaver()

agent = create_agent(
    model,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory # checkpointer의 인자로 전달하여 사용.
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 내 이름은 민수야. 잘 부탁해!"}]},
    {"configurable": {"thread_id": "1"}} # 사용자 관리를 위한 thread_id를 추가 정보로 제공한다.
)

print(response['messages'][-1].content)

안녕하세요, 민수님! 잘 부탁드립니다. 무엇을 도와드릴까요?


In [54]:
# 메모리 내용을 출력해 보면 대화 내용들이 모두 `memory` 변수에 잘 저장되어 있는 것을 확인할 수 있습니다.
for message in memory.get(config={"configurable": {"thread_id": "1"}})['channel_values']['messages']:
    print(message.content)

안녕! 내 이름은 민수야. 잘 부탁해!
안녕하세요, 민수님! 잘 부탁드립니다. 무엇을 도와드릴까요?


In [55]:
# 이제 개발자가 직접 이전 대화 기록들을 입력해 주지 않아도 이전 대화 내용을 기억합니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

민수님이세요.


`thread_id`를 이용해 여러 대화를 각각 관리하는 것도 훨씬 용이합니다!

In [56]:
# thread_id=1 : 민수 / thread_id=2 : 철수

response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 철수야. 오늘 날씨 어때?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

오늘 날씨는 맑아요.


In [57]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

철수예요.


`InMemorySaver()`는 단기 메모리입니다. 따라서 프로그램이 종료되면 사라지게 되죠.

이걸 프로그램이 종료된 뒤에도 기억하게 하기 위해선 해당 정보들을 따로 저장해야 합니다.

실제 서비스에서는 LangGraph의 Store와 DB를 연동해서 사용하지만 지금은 해당 정보들을 기록할 json 파일을 만들어 보도록 하겠습니다.

사용자에 대한 정보를 기록할 수 있는 txt 파일을 만든 뒤, tool을 이용해서 해당 파일을 관리할 겁니다!

In [ ]:
from langchain_core.runnables import RunnableConfig

@tool
def save_user_info(info: str) -> str:
    """사용자에 대한 정보를 기록합니다. 추후 대화에 도움이 될 만한 사용자 정보를 기록하는데 사용하세요."""
    
    with open(f"user_profile.txt", "w") as f:
        f.write(info)
    
    return "사용자 정보를 저장했습니다."

@tool
def load_user_info() -> str:
    """저장된 사용자 정보를 불러옵니다."""
    
    try:
        with open(f"user_profile.txt", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

In [20]:
agent = create_agent(
    model,
    tools=[save_user_info, load_user_info],
    checkpointer=InMemorySaver()
)

In [21]:
response = agent.invoke(
    {
        "messages": {"role": "user", "content": "안녕, 난 민희야. 앞으로 잘 부탁해!"},
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

안녕 민희야! 반가워 😊  
앞으로 잘 부탁해. 편하게 말 걸어줘!


`user_profile.txt` 파일을 확인해 봅시다!

#### Middleware

Middleware란, llm이 특정 작업을 처리하기 전/후로 개입할 수 있는 기능을 말합니다.

예를 들면, 대화가 너무 길어지면 자동으로 지난 대화를 요약한다거나, 위험할 수 있는 작업을 수행하기 전에 사람의 허락을 받게 한다던가 하는 것이죠.

자주 사용되는 기능들은 LangChain에서 이미 구현해 둔 middleware들도 존재하고, 직접 커스텀한 middleware도 사용 가능합니다.

##### SummarizationMiddleware

대화가 길어지게 되면 이전 대화 내용들의 토큰 수가 점점 부담될 수 있습니다.

이럴 땐 `SummarizationMiddleware`를 활용해 이전의 대화 기록을 요약할 수 있습니다!

* model : 요약을 수행할 모델의 이름.
* trigger : 요약을 진행할 조건.
    * tokens : 토큰 수를 기준으로
    * messages : 메세지 수를 기준으로
    * fraction : 모델 최대 컨텍스트 크기 대비 비율 (0~1.0)
* keep : 요약 후 얼마나 남길 지.
    * messages : 최근 n개 메세지만 유지
    * tokens : 최근 n개 토큰만 유지

요약 확인을 위해 `trigger`와 `keep`을 작게 설정해 보겠습니다.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model,
    # tools=tools,
    middleware=[
        SummarizationMiddleware(
            model=model_name,
            trigger=("tokens", 200), # 대화 기록의 총 토큰 수가 200을 넘어가면 자동으로 요약을 진행한다.
            keep=("messages", 2) # 최근 2개의 대화기록은 남김.
        )
    ],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver()
)

이제 200토큰이 넘어갈 때까지 대화를 한 뒤...

In [11]:
response = agent.invoke(
    {
        "messages": {"role": "user", "content": "일 하기 싫은데 나한테 문제가 있는걸까?"},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

response = agent.invoke(
    {
        "messages": {"role": "user", "content": "일하지 않고 돈을 벌고 싶어..."},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

response = agent.invoke(
    {
        "messages": {"role": "user", "content": "하루종일 자고 싶기도 해."},
        "user_name": "minhee0219",
        "prefer_language": "Korean"
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

그럴 수도 있지만, **“내가 이상한가?”로 바로 연결할 필요는 없어요.**  
일 하기 싫은 마음은 꽤 흔하고, 보통은 여러 이유가 섞여 있습니다.

가능한 이유들:
- **단순한 피로/번아웃**: 너무 오래 버텨서 에너지가 바닥난 상태
- **일이 안 맞음**: 흥미, 가치관, 강도, 환경이 안 맞을 수 있음
- **스트레스/불안**: 시작하기 전에 이미 부담이 커서 회피하게 됨
- **우울감/무기력**: 의욕 자체가 전반적으로 떨어진 상태
- **수면/건강 문제**: 몸이 지치면 마음도 같이 무거워짐
- **완벽주의/실패 두려움**: 하기 싫다기보다 시작이 무서운 경우

스스로 점검해볼 질문:
1. **일만 싫은지**, 아니면 요즘 **전반적으로 모든 게 다 싫은지**
2. 쉬어도 회복이 되는지, 아니면 **쉬어도 계속 지치는지**
3. “하기 싫다”가 **지루함**인지, **두려움**인지, **무기력**인지
4. 최근 몇 주~몇 달간 **수면, 식욕, 집중력**이 떨어졌는지

대체로:
- **일만 유독 싫다** → 업무/환경/역할이 안 맞을 가능성
- **생활 전반이 다 힘들다** → 번아웃, 우울, 건강 문제도 고려해볼 만함

당장 해볼 수 있는 것:
- 오늘 할 일을 **아주 작게 쪼개기**  
  예: “30분 집중” 대신 “문서 열기 → 제목 쓰기”
- **휴식이 필요한지** 먼저 확인하기
- 가능하면 **잠, 식사, 움직임**부터 챙기기
- 믿을 만한 사람에게 **그냥 힘들다고 말하기**
- 상태가 몇 주 이상 계속되거나 일상에 영향이 크면 **상담/진료**도 도움이 돼요

원하면 내가 같이  
**“그냥 귀찮은 건지, 번아웃인지, 우울감인지”**  
간단한 체크 질문으로 같이 구분해줄게요.
그 마음 정말 자연스러워요.  
솔직히 **“일은 싫고, 돈은 필요하고, 쉬고 싶다”**는 건 많은 사람이 한 번쯤 느껴요. 그래서 그 자체만으로 이상한 건 아니에요.

다만 현실적으로는 **“아무것도 하지 않고 계속 돈이 들어오는 상태”**는 거의 없고,

결과를 확인해 볼까요?

In [14]:
state = agent.get_state({"configurable": {"thread_id": "1"}}).values

for message in state['messages']:
    print(message.content)

Here is a summary of the conversation to date:

## SESSION INTENT
사용자는 “일하기 싫은데 나한테 문제가 있는걸까?”라는 감정적/심리적 고민에서 시작해, 이제는 “일하지 않고 돈을 벌고 싶어...”라는 욕구를 표현함. 핵심 목표는 노동 회피 욕구의 배경을 이해하고, 이를 비난 없이 다루면서 현실적인 방향을 탐색하는 것.

## SUMMARY
- 대화 초점은 사용자의 “일 하기 싫다”는 감정이 정상인지, 자기에게 문제가 있는지에 대한 불안에서 시작됨.
- 이에 대해 일 하기 싫은 감정은 흔하며, 곧바로 “내가 이상하다”로 연결할 필요는 없다고 공감적으로 답함.
- 가능한 원인으로 피로/번아웃, 일이 안 맞음, 스트레스/불안, 우울감/무기력, 수면·건강 문제, 완벽주의/실패 두려움 등을 제시함.
- 사용자가 스스로 점검할 수 있도록 “일만 싫은지 vs 전반적으로 다 싫은지”, “쉬어도 회복되는지”, “두려움/무기력/지루함 중 무엇인지”, “수면·식욕·집중력 변화가 있는지” 같은 질문을 제공함.
- 상황이 몇 주 이상 지속되거나 일상에 큰 영향을 주면 상담/진료도 도움이 될 수 있다고 안내함.
- 마지막 사용자 메시지에서 사용자는 “일하지 않고 돈을 벌고 싶어...”라고 말해, 단순한 직무 불만을 넘어 노동 자체에 대한 강한 회피와 경제적 독립 욕구를 드러냄.
- 아직 이에 대한 후속 응답은 없으며, 사용자의 욕구를 도덕적으로 판단하기보다 숨은 필요(휴식, 자유, 통제감, 탈진, 의미 상실 등)를 탐색하는 것이 적절함.

## ARTIFACTS
None

## NEXT STEPS
- 사용자의 “일하지 않고 돈을 벌고 싶다”는 말 뒤에 있는 감정과 필요를 공감적으로 탐색하기.
- 판단 없이 현실적 제약을 인정하면서, “왜 그렇게 느끼는지”를 함께 정리하기.
- 필요하면 비노동적 수입(자산/투자/자동화/부수입/프리랜스 최소화 등)에 대한 현실성, 기대치, 위험을 부드럽게 다루기.
- 사용자가 원하면 현재 상태가 번아웃